# The Linear Inverted Pendulum Model

In [1]:
import numpy as np
np.set_printoptions(linewidth=10000)

## Define states and controls
We select as states the position $\mathbf{r}\in\mathbb{R}^2$ and velocity $\dot{\mathbf{r}}\in\mathbb{R}^2$ of the Center of Mass (CoM) along the $XY$ directions.

We collect the states into a single state vector $\mathbf{x}$:

$\mathbf{x} = \begin{bmatrix}\mathbf{r} \\ \dot{\mathbf{r}} \end{bmatrix} \in \mathbb{R}^4$

In [2]:
# States
nr = 2 # CoM position size
nrdot = 2 # CoM velocity size

nx = nr + nrdot # State dimension

We use as control the Zero Moment Point (ZMP):

$\mathbf{u} = \mathbf{z} \in \mathbb{R}^2$

In [3]:
# Controls
nz = nu = 2 # ZMP position (control) dimension

## Optimal Control Problem Parameters
The parameters of the Optimal Control (OC) problem are:

- the number of nodes $ns$
- the final time $tf$

They are used to compute the time interval $dt = \frac{tf}{ns}$.

IMPORTANT: We consider $ns+1$ states and $ns$ controls. We start from a state $\mathbf{x}_k$ and apply the control $\mathbf{u}_k$ to end up in state $\mathbf{x}_{k+1}$, $\forall k \in \{0, ..., ns-1 \}$. There is no control applied at the final node.

In [4]:
ns = 40 # Number of nodes
tf = 3. # Total time in seconds 
dt = tf/ns

## Linear Inverted Pendulum Model Params
The equations of motion for the Linear Inverted Pendulum Model (LIPM) are:

$\ddot{\mathbf{r}} = \omega^2\left(\mathbf{r} - \mathbf{z}\right)$

with $\omega = \sqrt{\frac{9.81}{h}}$ and $h:$ the height of the CoM considered constant.

In [5]:
h = 0.83 # CoM height (constant)
w = np.sqrt(9.81/h)

## Define LIPM Dynamics
The dynamics of the system are defined as a double integrator:

$\dot{\mathbf{x}} = \begin{bmatrix} \dot{\mathbf{r}} \\ \ddot{\mathbf{r}} \end{bmatrix} = \begin{bmatrix} \dot{\mathbf{r}} \\ \omega^2 \left(\mathbf{r} - \mathbf{z} \right) \end{bmatrix} \in \mathbb{R}^4$, 

which is linear in the states and controls:

$\dot{\mathbf{x}} = \mathbf{A}\mathbf{x} + \mathbf{B}\mathbf{u} = \begin{bmatrix} \mathbf{0} & \mathbf{I} \\ \omega^2 \mathbf{I} & \mathbf{0}\end{bmatrix}\begin{bmatrix}\mathbf{r} \\ \dot{\mathbf{r}} \end{bmatrix} + \begin{bmatrix} \mathbf{0} \\ -\omega^2 \mathbf{I} \end{bmatrix}\mathbf{z}$

In [6]:
def dynamics(w):
    A = np.zeros((nx, nx))
    A[0:2, 2:4] = np.eye(2)
    A[2:4, 0:2] = w**2 * np.eye(2)

    B = np.zeros((nx, nu))
    B[2:4, 0:2] = -w**2 * np.eye(2)
    return A, B

## Cost Function
We consider a cost function made up of a sum of L2-norm terms. For the first $ns$ nodes, we wish to minimize the CoM velocity and the control action (ZMP). In the final node, we wish to minimize the CoM velocity.

This cost function can be written as a Quadratic Program (QP):

$F =  \frac{1}{2} \sum\limits_{k=0}^{ns-1}\left[Q\dot{\mathbf{r}}_k^T\dot{\mathbf{r}}_k + R\left( \mathbf{z}_k - \mathbf{z}_{r,k} \right)^T\left( \mathbf{z}_k - \mathbf{z}_{r,k} \right)\right] + Q_{fin}\dot{\mathbf{r}}_{ns}^T\dot{\mathbf{r}}_{ns} = $

$=\frac{1}{2}\begin{bmatrix}\mathbf{x}^T & \mathbf{u}^T \end{bmatrix}^T \mathbf{H}\begin{bmatrix}\mathbf{x} \\ \mathbf{u} \end{bmatrix} + \mathbf{g}^T\begin{bmatrix}\mathbf{x} \\ \mathbf{u} \end{bmatrix}$

The Hessian $\mathbf{H}$ and the gradient $\mathbf{g}$ have the following dimensions: 

$\mathbf{H} \in \mathbb{R}^{ns(nx + nu) + nx \times ns(nx + nu) + nx}$

$\mathbf{g} \in \mathbb{R}^{ns(nx + nu) + nx}$

In particular, the Hessian is block diagonal with the single block:

$\mathbf{H}_k = \begin{bmatrix}\mathbf{r}_k^T & \dot{\mathbf{r}}_k^T & \mathbf{u}_k^T \end{bmatrix} 
\begin{bmatrix}
\mathbf{0} & \mathbf{0} & \mathbf{0} \\
\mathbf{0} & Q\mathbf{I} & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & R\mathbf{I}
\end{bmatrix}
\begin{bmatrix}\mathbf{r}_k \\ \dot{\mathbf{r}}_k \\ \mathbf{u}_k \end{bmatrix}$,

the final row:

$\mathbf{H}_{ns} = \begin{bmatrix}\mathbf{r}_{ns}^T & \dot{\mathbf{r}}_{ns}^T \end{bmatrix} 
\begin{bmatrix}
\mathbf{0} & \mathbf{0} \\
\mathbf{0} & Q_{fin}\mathbf{I}
\end{bmatrix}
\begin{bmatrix}\mathbf{r}_{ns} \\ \dot{\mathbf{r}}_{ns} \end{bmatrix}$.

While the gradient:
$\mathbf{g}_k = \begin{bmatrix}\mathbf{0}^T  & \mathbf{0}^T & -R\mathbf{z}_{ref}^T  \end{bmatrix}\begin{bmatrix}\mathbf{r}_k \\ \dot{\mathbf{r}}_k \\ \mathbf{u}_k \end{bmatrix}$.

Notice: the reference enters in the gradient only.

In [7]:
# Cost Function Params
Q = 1e-3
Qfin = 1e6
R = 1e1

In [8]:
def running_cost_hessian(Q, R):
    """
    Hessian running cost for a single stage
    """
    H = np.zeros((nx + nu, nx + nu))
    H[2:4, 2:4] = Q * np.eye(2)
    H[4:6, 4:6] = R * np.eye(2)
    return H

In [9]:
def running_cost_gradient(R, zref):
    """
    Gradient running cost for single stage
    """
    g = np.zeros((nx + nu, 1))
    g[4:6,0] = -R * zref
    return g

In [10]:
def final_cost_hessian(Qfin):
    """
    Hessian final cost
    """
    H = np.zeros((nx, nx))
    H[2:4, 2:4] = Qfin * np.eye(2)
    return H

In [11]:
# Build the QP matrices
H = np.zeros((ns * (nx + nu) + nx, ns * (nx + nu) + nx))
for i in range(ns):
    H[i * (nx + nu):(i + 1) * (nx + nu), i * (nx + nu):(i + 1) * (nx + nu)] = running_cost_hessian(Q, R)
H[-nx:, -nx:] = final_cost_hessian(Qfin)

In [12]:
grad = np.zeros((ns * (nx + nu) + nx, 1))
for i in range(ns):
    grad[i * (nx + nu):(i + 1) * (nx + nu)] = running_cost_gradient(R, np.zeros((2,1)).flatten())

## Constraints

### Euler Integration: 
Let's impose the system dynamics (double integrator). We'll use Euler integration for the discretization:

$\mathbf{x}_{k+1} = \mathbf{x}_{k} + \dot{\mathbf{x}}_{k}dt = \mathbf{x}_{k} + dt\left(\mathbf{Ax}_k + \mathbf{Bu}_k \right) = \left( \mathbf{I} + dt\mathbf{A}\right)\mathbf{x}_k + dt\mathbf{Bu}_k$

The linear constraint can be written:

$\left( \mathbf{I} + dt\mathbf{A}\right)\mathbf{x}_k + dt\mathbf{Bu}_k - \mathbf{x}_{k+1} = \mathbf{0}$

For a single stage (node) in matrix form:

$\begin{bmatrix}dt\mathbf{A}+\mathbf{I} & dt\mathbf{B} & -\mathbf{I} \end{bmatrix}\begin{bmatrix}\mathbf{x}_k \\ \mathbf{u}_k \\ \mathbf{x}_{k+1}\end{bmatrix}$,

This constraint connects $\mathbf{x}_k$ with $\mathbf{x}_{k+1}$ through the applied control $\mathbf{u}_k$.

The final constraint has the form:

$\mathbf{E}\begin{bmatrix}\mathbf{x}_0 \\ \mathbf{u}_0 \\ \dots \\ \mathbf{x}_{ns-1} \\ \mathbf{u}_{ns-1} \\ \mathbf{x}_{ns}\end{bmatrix} = \mathbf{0}$

In [13]:
def Euler(A, B, dt):
    """
    Euler integration for a single stage (node)
    """
    E = np.zeros((nx, 2 * nx + nu))
    E[:, 0:nx] = np.eye(nx) + dt * A
    E[:, nx:nx + nu] = dt * B
    E[:, nx + nu:] = -np.eye(nx)
    return E

In [14]:
# Integrate over the full horizon
E = np.zeros((ns * nx, ns * (nx + nu) + nx))
A, B = dynamics(w)
for i in range(ns):
    E[i * nx:(i + 1) * nx, i * (nx + nu):(i + 1) * (nx + nu) + nx] = Euler(A, B, dt)

e = np.zeros((ns * nx, 1))

### Initial state
We want to impose the value of the initial state:

$\mathbf{x}_0 = \mathbf{x}_m$,

which can be written in matrix form:

$\begin{bmatrix}\mathbf{I} & \mathbf{0} & \dots & \mathbf{0} \end{bmatrix} \begin{bmatrix}\mathbf{x}_0 \\ \mathbf{x}_1 \\ \vdots \\ \mathbf{x}_{ns+1}\end{bmatrix} = \mathbf{x}_m$

In [15]:
# Initial state constraint
x0 = np.zeros((nx,1))
e = np.concatenate([e, x0])
init_state_constraint = np.zeros((nx, ns * (nx + nu) + nx))
init_state_constraint[:, :nx] = np.eye(nx)
E = np.vstack((E, init_state_constraint))